# Paso 2 — Cargar el Staging (Bronze → Silver)
**Archivo original:** `etl/ETL-Cargar-Staging.py`

---

## ¿Qué hace este script?

Lee los archivos Excel de la capa **Bronze** y los carga en las tablas de **Staging** (Silver).
Es la primera transformación: los datos pasan de Excel desordenado a una tabla SQL estructurada.

Tareas principales:
1. Leer `DatosXLSX.xlsx` con Pandas
2. Convertir las columnas de fecha de texto a tipo DATE
3. Renombrar las 41 columnas al estándar del DW
4. Insertar en `dw.stg_aduana`
5. Leer `LISTADO_DE_DESTINACIONES.xlsx` e insertar en `dw.stg_destinaciones`


---

## Bloque 1: Importaciones y rutas


In [ ]:
import duckdb   # para conectarse al DW
import pandas as pd  # para leer Excel y manipular datos
import os       # para verificar existencia de archivos
import re       # para expresiones regulares (normalización de columnas)

DB_PATH   = r"C:\Información\proyectos\aduana_bi\db\aduana.duckdb"
XLSX_PATH = r"C:\Información\proyectos\aduana_bi\data_lake\bronze\DatosXLSX.xlsx"
DEST_PATH = r"C:\Información\proyectos\aduana_bi\data_lake\bronze\LISTADO_DE_DESTINACIONES.xlsx"

print("Rutas configuradas.")

---

## Bloque 2: Conexión y limpieza del staging

Antes de cargar datos nuevos, borramos los anteriores con `DELETE`.
Esto permite ejecutar el script múltiples veces sin duplicar filas.


In [ ]:
con = duckdb.connect(DB_PATH)

try:
    # DELETE sin WHERE borra todas las filas, pero mantiene la estructura de la tabla
    # (diferente a DROP TABLE que borra la tabla entera)
    con.execute("DELETE FROM dw.stg_aduana;")
    con.execute("DELETE FROM dw.stg_destinaciones;")
    print("Staging limpiado.")
except:
    # Si las tablas no existen todavía, simplemente se ignora el error
    print("Tablas no existen aún, se omite DELETE.")

---

## Bloque 3: Leer el Excel con Pandas

`pd.read_excel()` lee el archivo y lo convierte en un **DataFrame** (tabla en memoria).

**¿Por qué usamos Pandas y no cargamos el Excel directo a DuckDB?**
Porque las columnas de fecha vienen como texto en formato `'DD/MM/YYYY'`.
Pandas nos permite convertirlas antes de insertar en la base de datos.


In [ ]:
df = pd.read_excel(XLSX_PATH)

print(f"Filas leídas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
print("\nPrimeras columnas del Excel:")
print(list(df.columns[:5]))

---

## Bloque 4: Conversión de fechas

Las fechas en el Excel están como texto: `'15/03/2024'`.
Las convertimos a objetos DATE de Python con `pd.to_datetime()`.

- `format='%d/%m/%Y'`: le decimos a Pandas el formato exacto del texto
- `errors='coerce'`: si hay un valor que no puede convertirse, lo convierte a `NaT` (null de fecha) en lugar de lanzar un error
- `.dt.date`: extrae solo la parte de fecha (sin hora)


In [ ]:
df['OFICIALIZACION'] = pd.to_datetime(
    df['OFICIALIZACION'],
    format='%d/%m/%Y',
    errors='coerce'
).dt.date

df['CANCELACION'] = pd.to_datetime(
    df['CANCELACION'],
    format='%d/%m/%Y',
    errors='coerce'
).dt.date

# Verificar que no hayan quedado fechas nulas
nulls_ofic = df['OFICIALIZACION'].isna().sum()
nulls_canc = df['CANCELACION'].isna().sum()

print(f"Fechas OFICIALIZACION NULL: {nulls_ofic}")
print(f"Fechas CANCELACION NULL:    {nulls_canc}")

# Si el 100% son NULL, hay un problema con el formato del Excel
if nulls_ofic == len(df):
    raise ValueError("ERROR CRÍTICO: 100% de OFICIALIZACION son NULL. Verificar formato.")

---

## Bloque 5: Renombrar columnas

El Excel tiene columnas con espacios, mayúsculas y caracteres especiales:
`'DESPACHO CIFRADO'`, `'AÑO'`, `'PAIS ORIGEN'`...

En SQL se prefieren nombres en minúsculas con guiones bajos:
`despacho_cifrado`, `anio`, `pais_origen`...

El diccionario mapea cada nombre original al nombre destino.


In [ ]:
df = df.rename(columns={
    'DESPACHO CIFRADO'          : 'despacho_cifrado',
    'OPERACION'                 : 'operacion',
    'DESTINACION'               : 'destinacion',
    'REGIMEN'                   : 'regimen',
    'OFICIALIZACION'            : 'oficializacion',
    'CANCELACION'               : 'cancelacion',
    'AÑO'                       : 'anio',
    'MES'                       : 'mes',
    'ADUANA'                    : 'aduana',
    'COTIZACION'                : 'cotizacion',
    'MEDIO TRANSPORTE'          : 'medio_transporte',
    'CANAL'                     : 'canal',
    'ITEM'                      : 'item',
    'PAIS ORIGEN'               : 'pais_origen',
    'PAIS PROCEDENCIA/DESTINO'  : 'pais_procedencia_destino',
    'USO'                       : 'uso',
    'UNIDAD MEDIDA ESTADISTICA' : 'unidad_medida_estadistica',
    'CANTIDAD ESTADISTICA'      : 'cantidad_estadistica',
    'KILO NETO'                 : 'kilo_neto',
    'KILO BRUTO'                : 'kilo_bruto',
    'FOB DOLAR'                 : 'fob_dolar',
    'FLETE DOLAR'               : 'flete_dolar',
    'SEGURO DOLAR'              : 'seguro_dolar',
    'IMPONIBLE DOLAR'           : 'imponible_dolar',
    'IMPONIBLE GS'              : 'imponible_gs',
    'AJUSTE A INCLUIR'          : 'ajuste_a_incluir',
    'AJUSTE A DEDUCIR'          : 'ajuste_a_deducir',
    'POSICION'                  : 'posicion',
    'RUBRO'                     : 'rubro',
    'DESC CAPITULO'             : 'desc_capitulo',
    'DESC POSICION'             : 'desc_posicion',
    'DESC PARTIDA'              : 'desc_partida',
    'MERCADERIA'                : 'mercaderia',
    'MARCA ITEM'                : 'marca_item',
    'ACUERDO'                   : 'acuerdo',
    'DERECHO'                   : 'derecho',
    'ISC'                       : 'isc',
    'SERVICIO'                  : 'servicio',
    'RENTA'                     : 'renta',
    'IVA'                       : 'iva',
    'OTROS'                     : 'otros',
    'TOTAL'                     : 'total',
})

print("Columnas renombradas. Primeras 5:")
print(list(df.columns[:5]))

---

## Bloque 6: Insertar en stg_aduana

`con.register('nombre', df)` registra el DataFrame de Pandas como si fuera una tabla SQL temporal.
Después podemos hacer `SELECT ... FROM df_stg` directamente en SQL.

El `INSERT INTO ... SELECT ...` con `CAST()` garantiza que los tipos de datos sean correctos
al pasar de Pandas (que tiene sus propios tipos) a DuckDB.


In [ ]:
# Registrar el DataFrame como tabla virtual en DuckDB
con.register('df_stg', df)

# Insertar con casteo explícito de tipos
con.execute("""
INSERT INTO dw.stg_aduana
SELECT
    CAST(despacho_cifrado           AS VARCHAR),
    CAST(operacion                  AS VARCHAR),
    CAST(destinacion                AS VARCHAR),
    CAST(regimen                    AS VARCHAR),
    CAST(oficializacion             AS DATE),
    CAST(cancelacion                AS DATE),
    CAST(anio                       AS INTEGER),
    CAST(mes                        AS VARCHAR),
    CAST(aduana                     AS VARCHAR),
    CAST(cotizacion                 AS DOUBLE),
    CAST(medio_transporte           AS VARCHAR),
    CAST(canal                      AS VARCHAR),
    CAST(item                       AS INTEGER),
    CAST(pais_origen                AS VARCHAR),
    CAST(pais_procedencia_destino   AS VARCHAR),
    CAST(uso                        AS VARCHAR),
    CAST(unidad_medida_estadistica  AS VARCHAR),
    CAST(cantidad_estadistica       AS DOUBLE),
    CAST(kilo_neto                  AS DOUBLE),
    CAST(kilo_bruto                 AS DOUBLE),
    CAST(fob_dolar                  AS DOUBLE),
    CAST(flete_dolar                AS DOUBLE),
    CAST(seguro_dolar               AS DOUBLE),
    CAST(imponible_dolar            AS DOUBLE),
    CAST(imponible_gs               AS DOUBLE),
    CAST(ajuste_a_incluir           AS DOUBLE),
    CAST(ajuste_a_deducir           AS DOUBLE),
    CAST(posicion                   AS VARCHAR),
    CAST(rubro                      AS VARCHAR),
    CAST(desc_capitulo              AS VARCHAR),
    CAST(desc_posicion              AS VARCHAR),
    CAST(desc_partida               AS VARCHAR),
    CAST(mercaderia                 AS VARCHAR),
    CAST(marca_item                 AS VARCHAR),
    CAST(acuerdo                    AS VARCHAR),
    CAST(derecho                    AS DOUBLE),
    CAST(isc                        AS DOUBLE),
    CAST(servicio                   AS DOUBLE),
    CAST(renta                      AS DOUBLE),
    CAST(iva                        AS DOUBLE),
    CAST(otros                      AS DOUBLE),
    CAST(total                      AS DOUBLE)
FROM df_stg;
""")

filas = con.execute("SELECT COUNT(*) FROM dw.stg_aduana").fetchone()[0]
print(f"stg_aduana cargada: {filas} filas")

---

## Bloque 7: Cargar el catálogo de destinaciones

El archivo `LISTADO_DE_DESTINACIONES.xlsx` es un catálogo auxiliar con la descripción
de cada código de destinación aduanera (ej: `IM4 = Importación definitiva al consumo`).

Las columnas de este Excel tienen caracteres especiales, por eso se normaliza primero:
- `c.strip().upper()`: elimina espacios y pasa a mayúsculas
- `re.sub(r'^[^A-Z0-9]+', '', ...)`: elimina caracteres no alfanuméricos del inicio


In [ ]:
if os.path.exists(DEST_PATH):
    try:
        df_dest = pd.read_excel(DEST_PATH)

        # Normalizar nombres de columnas: quitar BOM, acentos iniciales, espacios
        df_dest.columns = [
            re.sub(r'^[^A-Z0-9]+', '', c.strip().upper())
            for c in df_dest.columns
        ]

        print("Columnas del Excel de destinaciones:", df_dest.columns.tolist())

        # Mapear nombres del Excel al estándar del DW
        df_dest = df_dest.rename(columns={
            "CÓD."                              : "cod_destinacion",
            "COD."                              : "cod_destinacion",
            "DESCRIPCIÓN"                       : "descripcion_dest",
            "DESCRIPCION"                       : "descripcion_dest",
            "SUSPENSIVO - DEFINITIVO - TEMPORAL": "tipo_regimen_base",
            "IMPORT / EXPORT"                   : "tipo_operacion_base"
        })

        # Si alguna columna esperada no existe, crearla con valor nulo
        if "tipo_regimen_base"   not in df_dest.columns: df_dest["tipo_regimen_base"]   = None
        if "tipo_operacion_base" not in df_dest.columns: df_dest["tipo_operacion_base"] = None

        # Seleccionar solo las columnas que necesita la tabla destino
        df_dest = df_dest[["cod_destinacion", "descripcion_dest",
                           "tipo_regimen_base", "tipo_operacion_base"]]

        con.register("df_dest", df_dest)
        con.execute("INSERT INTO dw.stg_destinaciones SELECT * FROM df_dest;")
        print("stg_destinaciones cargada correctamente.")

    except Exception as e:
        print(f"Error cargando Excel de destinaciones: {e}")
else:
    print("Aviso: no se encontró LISTADO_DE_DESTINACIONES.xlsx. Se continúa sin catálogo.")

---

## Bloque 8: Cerrar y verificar


In [ ]:
con.close()
print("Staging cargado correctamente.")

In [ ]:
# Verificación rápida: ver las primeras filas del staging
con = duckdb.connect(DB_PATH)

print("=== Muestra de stg_aduana ===")
muestra = con.execute("""
    SELECT despacho_cifrado, operacion, destinacion, oficializacion, fob_dolar
    FROM dw.stg_aduana
    LIMIT 5
""").fetchdf()
print(muestra.to_string())

print("\n=== stg_destinaciones (primeras 3) ===")
dest = con.execute("SELECT * FROM dw.stg_destinaciones LIMIT 3").fetchdf()
print(dest.to_string())

con.close()

---

**Resultado esperado:** `stg_aduana` con N filas (según el Excel), `stg_destinaciones` con los códigos del catálogo.

**Siguiente paso:** `03_Cargar_Dimensiones.ipynb`
